# Influence of `noise_viab` on signal fidelity and MLP denoising, at fixed `mu=50`, `T_viab=0.8`, `d0=200,000`

`noise_viab` is the per-cell expression noise in `produce_capsids`
(`E_{s,j} = exp(noise_viab * Z_{s,j})`, `sequence_classesV1.py`): it sits on top of the
score-driven capsid-production rate and is the ONLY parameter swept here. `mu = rho * N1 / d0`
is held fixed at `50` (deep in `mu_HEK_multiplicity_sweep.ipynb`'s signal-fidelity plateau) and
`T_viab` fixed at `0.8` (between the two peaks -- Pearson `r` peaked at `T_viab~1`, top-1000
recovery peaked at `T_viab~0.5` -- found in `T_viab_sweep.ipynb`). The pool itself is also much
larger than the sibling notebooks in this folder: `d0=200,000` instead of `20,000`.

**Question.** As `noise_viab` grows, does the `ProfileMLP` genuinely denoise -- i.e. does its
prediction stay CLOSER to the true noiseless GT score than the raw NGS-measured `target1` label
it was trained on, both in Pearson `r` and in top-1000 recovery? This is the same question
`MLP_viability_noise_denoising20K.ipynb`/`50K.ipynb` ask, rebuilt on this folder's current
recipe (`mu`/`T_viab` fixed at their own found-good values, `multinomialNGS=True`, `topk_recovery`
+ Pearson `r` as the twin metrics established in `T_viab_sweep.ipynb`).

## 0. Setup

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Must run before the first `import jax` anywhere -- same convention as the sibling notebooks.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from flax import nnx
import optax

from sequence_classesV1 import *
from analysisV1 import *
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
from initialize_weights import load_F_viab_aav9_potts, load_J_viab_aav9_potts, NUM_AMINO_ACIDS, NUM_POSITIONS

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

## 1. Ground truth and fixed pool

Identical real AAV9 `F_viab`/`J_viab` and permuted `F_sel`/`J_sel` as the sibling notebooks
(`F_sel`/`J_sel` are constructed but never used below -- only the viability channel, `target1`,
is swept here). The pool itself is `d0=200,000` random 7-mers, 10x larger than
`mu_HEK_multiplicity_sweep.ipynb`/`T_viab_sweep.ipynb`.

In [ ]:
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
F_viab = load_F_viab_aav9_potts()
J_viab = load_J_viab_aav9_potts()

key_F, key_J = jax.random.split(jax.random.key(0), 2)
sigma_F = jax.random.permutation(key_F, NUM_AMINO_ACIDS)
F_sel   = F_viab[sigma_F, :]
sigma_J = jax.random.permutation(key_J, NUM_AMINO_ACIDS)
J_sel   = J_viab[:, :, sigma_J, :][:, :, :, sigma_J]

key_pool = jax.random.key(1)
N = 200_000
sequences = jax.random.randint(key_pool, shape=(N, NUM_POSITIONS), minval=0, maxval=NUM_AMINO_ACIDS)
d0 = N

print(f"F_viab shape: {F_viab.shape}   J_viab shape: {J_viab.shape}   sequences: {N:,}")

## 2. Fixed 50,000-sequence evaluation pool (SAME key in every notebook)

A separate, FIXED `50,000`-sequence pool, generated once with a hardcoded key
(`EVAL_POOL_KEY_SEED=999`, `EVAL_POOL_SIZE=50_000` -- the exact same pair of values used in
`deeper_mlp/diversity_sweep_deeper_mlp.ipynb` and every other notebook that wants directly
comparable top-K recovery numbers), held out from training entirely. Every model trained at
every `noise_viab` gets evaluated on this SAME `50,000` sequences (in addition to this
notebook's own in-sweep test fold), so top-1000 recovery becomes comparable both across
`noise_viab` within this notebook AND across every other sweep notebook using the same fixed
key -- independent of this notebook's own pool size or train/test split.

In [ ]:
EVAL_POOL_KEY_SEED = 999   # fixed across every notebook -- reuse this exact value for comparable numbers
EVAL_POOL_SIZE     = 50_000

def compute_score_array(seq, F, J, L=NUM_POSITIONS):
    """Noiseless ground-truth score, decoupled from any Protocol instance -- same formula as
    Protocol.compute_score, applied directly to an arbitrary sequence array."""
    scores = jnp.sum(F[seq, jnp.arange(L)], axis=1)
    for i in range(L):
        for j in range(i + 1, L):
            scores = scores + J[i, j, seq[:, i], seq[:, j]]
    return scores

eval_sequences = jax.random.randint(jax.random.key(EVAL_POOL_KEY_SEED),
                                     shape=(EVAL_POOL_SIZE, NUM_POSITIONS),
                                     minval=0, maxval=NUM_AMINO_ACIDS)
eval_viab_score = np.asarray(compute_score_array(eval_sequences, F_viab, J_viab))
X_eval = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[np.asarray(eval_sequences)].reshape(EVAL_POOL_SIZE, -1)

print(f"fixed evaluation pool: {EVAL_POOL_SIZE:,} sequences (key seed={EVAL_POOL_KEY_SEED})")
print(f"GT score range on eval pool: [{eval_viab_score.min():.2f}, {eval_viab_score.max():.2f}]")

## 3. Fixed `mu=50`, `T_viab=0.8`, `noise_viab` grid, and train/test split

`rho=RHO_REF` (same baseline as the sibling notebooks), `N1` solved so that
`rho * N1 / d0 == 50` exactly -- note `N1` comes out to `1e10`, ABOVE `N0=1e9`, same regime
already exercised at high `mu` in `mu_HEK_multiplicity_sweep.ipynb` section 7.6 (`Poisson`
sampling doesn't require `N1 <= N0`). `NOISE_GRID` matches
`MLP_viability_noise_denoising50K.ipynb` for direct comparability.

In [ ]:
RHO_REF      = 1e-3
MU_FIXED     = 50
N1_FIXED     = MU_FIXED * d0 / RHO_REF
T_VIAB_FIXED = 0.8

NOISE_GRID = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
eps = 1.0

protocol = ProtocolV3(multinomialNGS=True, N0=N1_FIXED*10, N1=N1_FIXED,
        dilution_factor=10, sequences=sequences, D=1e9,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
        noise_viab=0.5, noise_sel=0.5, T_sel=1, T_viab=T_VIAB_FIXED,
        )
protocol._rho = float(RHO_REF)
viab_score = np.array(protocol.compute_score(F_viab, J_viab))
original_noise_viab = protocol.noise_viab

print(f"mu = {protocol._rho * protocol.N1 / d0:.2f}   T_viab = {protocol._T_viab}   N1 = {protocol.N1:.3g}")
print(f"noise_viab grid: {NOISE_GRID}")

In [ ]:
K_TOPK = 1000

X_all = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[np.asarray(sequences)].reshape(N, -1)
idx_train, idx_test = train_test_split(np.arange(N), test_size=0.5, random_state=0)
X_train_full, X_test = X_all[idx_train], X_all[idx_test]
viab_score_test = viab_score[idx_test]
k_frac_test = K_TOPK / len(idx_test)

print(f"pool: {N:,}   train: {len(idx_train):,}   test: {len(idx_test):,}   "
      f"top-{K_TOPK} = top-{100 * k_frac_test:.2f}% of the test set")

## 4. Simulate the SAME experimental configuration on the fixed eval pool

To get a genuine `GT<->protocol` recovery number on the fixed `50,000`-sequence eval pool
(section 2), build a SEPARATE `ProtocolV3` object dedicated to it alone -- NOT mixed into the
training pool (that would dilute `D`/`d0` and distort `mu`). `N1` is recomputed for the eval
pool's own `d0=50,000` to keep the same `mu`. Same reused-object / advancing-PRNG-key convention
as `protocol` itself: `protocol_eval.noise_viab` gets mutated to match every sweep point, right
alongside `protocol.noise_viab`, in section 6's loop.

In [ ]:
N1_eval = MU_FIXED * EVAL_POOL_SIZE / RHO_REF

protocol_eval = ProtocolV3(multinomialNGS=True, N0=N1_eval*10, N1=N1_eval,
        dilution_factor=10, sequences=eval_sequences, D=1e9,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
        noise_viab=0.5, noise_sel=0.5, T_sel=1, T_viab=1,
        )
protocol_eval._rho = float(RHO_REF)

print(f"mu (eval pool) = {protocol_eval._rho * protocol_eval.N1 / EVAL_POOL_SIZE:.2f}")

## 5. `ProfileMLP`: same architecture as the sibling notebooks, bigger batches

Same `ProfileMLP` (Linear + BatchNorm + Dropout + gelu, twice, then a scalar linear head) and
warmup-cosine-decay AdamW / early-stopping training loop as `T_viab_sweep.ipynb`. `batch_size`
is raised to `2048` (from `256`) -- the train fold here is `100,000` sequences, 10x the sibling
notebooks', so the default batch size would mean 10x more optimizer steps per epoch for no
real benefit; `2048` keeps steps-per-epoch (and wall-clock per epoch) in the same ballpark.

In [ ]:
def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


class ProfileMLP(nnx.Module):
    """
    MLP over the one-hot encoded per-position sequence (L=7 positions x A=20 amino acids
    -> 140 indicator features) -> scalar score. Identical to the sibling notebooks' ProfileMLP
    (Linear + BatchNorm + Dropout + gelu).
    """

    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (128, 64),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x, train: bool, rngs: nnx.Rngs = None):
        x = self.linear1(x)
        self.batchnorm1.use_running_average = not train
        x = self.batchnorm1(x)
        x = nnx.gelu(x)
        x = self.dropout1(x, rngs=rngs) if train else x
        x = self.linear2(x)
        self.batchnorm2.use_running_average = not train
        x = self.batchnorm2(x)
        x = nnx.gelu(x)
        x = self.dropout2(x, rngs=rngs) if train else x
        x = self.linear3(x)
        return x[:, 0]

In [ ]:
@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    """
    x : (batch, L*A) one-hot encoded per-position amino-acid indicators
    y : (batch,) target log enrichment
    """
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    y_pred = model(x, train=False)
    return jnp.mean((y_pred - y) ** 2)


@nnx.jit
def predict_log_enrichment(model, x):
    return model(x, train=False)


def train_profile_mlp(X_train, y_train, X_val, y_val, hidden_dims=(128, 64),
                       dropout_rate=0.1, epochs=300, batch_size=2048,
                       peak_lr=1e-3, final_lr=1e-5, weight_decay=0,
                       patience=20, seed=0, verbose=True):
    rngs = nnx.Rngs(seed)
    model = ProfileMLP(input_dim=X_train.shape[1], hidden_dims=hidden_dims,
                        dropout_rate=dropout_rate, rngs=rngs)

    n_train         = X_train.shape[0]
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps     = steps_per_epoch * epochs

    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr,
        warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9),
        end_value=final_lr,
    )
    optimizer = nnx.Optimizer(
        model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param
    )

    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val,   y_val   = jnp.asarray(X_val),   jnp.asarray(y_val)

    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs = float("inf"), None, 0

    for epoch in range(epochs):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm = jax.random.permutation(perm_key, n_train)
        Xs, ys = X_train[perm], y_train[perm]

        for start in range(0, n_train - n_train % batch_size, batch_size):
            xb = Xs[start:start + batch_size]
            yb = ys[start:start + batch_size]
            train_step(model, optimizer, xb, yb, rngs)

        val_loss = float(eval_step(model, X_val, y_val))
        if val_loss < best_val - 1e-6:
            best_val, bad_epochs = val_loss, 0
            best_state = nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"  early stop at epoch {epoch}, best val MSE = {best_val:.4f}")
                break

    if best_state is not None:
        nnx.update(model, best_state)
    return model, best_val

## 6. Sweep: one fresh `loop_DE()` + one fresh `ProfileMLP` per `noise_viab`

For each `noise_viab`: mutate `protocol.noise_viab` in place (same reused `protocol` object /
advancing-PRNG-key convention as the mu and T_viab sweeps), reset `lambda0`, run one `loop_DE()`
round to get `target1`, train a fresh `ProfileMLP` on the train fold, then compare on the
held-out test fold: Pearson `r` and top-`K_TOPK` recovery, each across the three pairwise views
(GT, protocol, MLP). `protocol.noise_viab` is restored to its original value afterward. At
`d0=200,000` the biological simulation itself (PCR/NGS steps in `loop_DE`) dominates wall-clock
time per point, not the MLP training.

In [ ]:
records = []

try:
    for nv in tqdm(NOISE_GRID, desc="noise_viab sweep"):
        protocol.noise_viab = float(nv)
        protocol.lambda0 = jnp.full(protocol.d0, protocol.N0 / protocol.d0)
        protocol_eval.noise_viab = float(nv)
        protocol_eval.lambda0 = jnp.full(protocol_eval.d0, protocol_eval.N0 / protocol_eval.d0)

        _bio_row, ngs_row = protocol.loop_DE()
        lambda0p, lambda2p, _lambda3p = (np.asarray(a) for a in ngs_row)
        target1 = np.log((lambda2p + eps) / (lambda0p + eps))
        _bio_row_eval, ngs_row_eval = protocol_eval.loop_DE()
        lambda0p_eval, lambda2p_eval, _lambda3p_eval = (np.asarray(a) for a in ngs_row_eval)
        target1_eval = np.log((lambda2p_eval + eps) / (lambda0p_eval + eps))
        target1_train, target1_test = target1[idx_train], target1[idx_test]

        Xtr, ytr, Xva, yva = split_train_val(X_train_full, target1_train, val_frac=0.15, seed=0)
        model, val_mse = train_profile_mlp(Xtr, ytr, Xva, yva, epochs=300, patience=20, verbose=False)
        pred_test = np.asarray(predict_log_enrichment(model, jnp.asarray(X_test)))
        pred_eval = np.asarray(predict_log_enrichment(model, jnp.asarray(X_eval)))

        records.append(dict(
            noise_viab=nv,
            r_gt_protocol=pearson(viab_score_test, target1_test),
            r_gt_mlp=pearson(viab_score_test, pred_test),
            r_protocol_mlp=pearson(target1_test, pred_test),
            rec_gt_protocol=topk_recovery(viab_score_test, target1_test, k=K_TOPK),
            rec_gt_mlp=topk_recovery(viab_score_test, pred_test, k=K_TOPK),
            rec_protocol_mlp=topk_recovery(target1_test, pred_test, k=K_TOPK),
            rec_gt_mlp_eval=topk_recovery(eval_viab_score, pred_eval, k=K_TOPK),
            r_gt_protocol_eval=pearson(eval_viab_score, target1_eval),
            rec_gt_protocol_eval=topk_recovery(eval_viab_score, target1_eval, k=K_TOPK),
            r_protocol_mlp_eval=pearson(target1_eval, pred_eval),
            rec_protocol_mlp_eval=topk_recovery(target1_eval, pred_eval, k=K_TOPK),
        ))
finally:
    protocol.noise_viab = original_noise_viab
    protocol_eval.noise_viab = original_noise_viab

noise_sweep_df = pd.DataFrame(records)
noise_sweep_df

## 7. Pearson correlation vs `noise_viab`

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(noise_sweep_df["noise_viab"], noise_sweep_df["r_gt_protocol"],  "o-", label="GT <-> protocol")
ax.plot(noise_sweep_df["noise_viab"], noise_sweep_df["r_gt_mlp"],       "o-", label="GT <-> MLP")
ax.plot(noise_sweep_df["noise_viab"], noise_sweep_df["r_protocol_mlp"], "o-", label="protocol <-> MLP")
ax.set_ylim(0, 1)
ax.set_xlabel("noise_viab")
ax.set_ylabel("Pearson r (on the log enrichments)")
ax.set_title(f"Pearson r vs noise_viab, mu={MU_FIXED}, T_viab={T_VIAB_FIXED} fixed, d0={d0:,}")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## 8. Top-1000 recovery vs `noise_viab` (in-sweep test fold)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(noise_sweep_df["noise_viab"], 100 * noise_sweep_df["rec_gt_protocol"],  "o-", label="GT <-> protocol")
ax.plot(noise_sweep_df["noise_viab"], 100 * noise_sweep_df["rec_gt_mlp"],       "o-", label="GT <-> MLP")
ax.plot(noise_sweep_df["noise_viab"], 100 * noise_sweep_df["rec_protocol_mlp"], "o-", label="protocol <-> MLP")
ax.set_ylim(0, 100)
ax.set_xlabel("noise_viab")
ax.set_ylabel(f"Top-{K_TOPK} recovery (%)")
ax.set_title(f"Top-{K_TOPK} recovery vs noise_viab, mu={MU_FIXED}, T_viab={T_VIAB_FIXED} fixed, d0={d0:,}")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Top-1000 recovery on the FIXED 50,000-sequence eval pool vs `noise_viab`

Same `50,000` sequences (section 2) at every `noise_viab` -- comparable across the whole
grid AND across every other notebook using the same `EVAL_POOL_KEY_SEED=999`/`EVAL_POOL_SIZE=50_000`
pair (e.g. `deeper_mlp/diversity_sweep_deeper_mlp.ipynb`). `GT<->protocol` (gray) comes from section 4's dedicated `protocol_eval` simulation,
mutated to the same `noise_viab` as `protocol` at every sweep point (section 6) -- the
natural reference for whether the MLP actually beats the raw protocol measurement on
this common benchmark.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(noise_sweep_df["noise_viab"], 100 * noise_sweep_df["rec_gt_protocol_eval"], "o-", color="gray", label="GT <-> protocol (fixed eval pool)")
ax.plot(noise_sweep_df["noise_viab"], 100 * noise_sweep_df["rec_gt_mlp_eval"], "o-", color="tab:red", label="GT <-> MLP (fixed eval pool)")
ax.axhline(100 * K_TOPK / EVAL_POOL_SIZE, color="black", lw=1, ls="--", label="random baseline")
ax.set_ylim(0, 100)
ax.set_xlabel("noise_viab")
ax.set_ylabel(f"Top-{K_TOPK} recovery on the fixed {EVAL_POOL_SIZE:,}-sequence eval pool (%)")
ax.set_title("GT <-> MLP recovery on a COMMON, fixed evaluation population")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## How to read this

- **Denoising signature**: if `GT <-> MLP` (orange) stays ABOVE `GT <-> protocol` (blue) as
  `noise_viab` grows -- in either Pearson `r` or top-K recovery, they need not agree, see
  `T_viab_sweep.ipynb`'s closing note -- the MLP is genuinely denoising: it recovers the true
  score better than the single noisy NGS measurement it was trained on, by averaging over the
  redundancy in its `100,000`-sequence training fold. If the orange curve instead tracks the
  blue one downward and stays below or equal to it, the model is just mirroring the noisy label
  with no real denoising benefit.
- **`protocol <-> MLP` (green) is expected to be the LOWEST of the three at every `noise_viab`**:
  the model's prediction never sees the test fold's specific noise draw (a fresh `loop_DE()` call
  from the training simulation), so it can only correlate with `target1_test` through whatever it
  learned about the true score -- adding independent noise to one side of a correlation can only
  hurt it, never help (see the discussion in this project's notes on why `MLP<->GT > MLP<->protocol`
  is the expected, not surprising, ordering).
- At `noise_viab=0` there is still measurement noise left (Poisson capsid counts, NGS read depth,
  `multinomialNGS=True` sampling) -- `GT <-> protocol` is not expected to reach `r=1` / 100%
  recovery even there.
- **Section 9 (fixed eval pool)** repeats the `GT<->MLP` recovery comparison on the SAME `50,000`-sequence population used by every other sweep notebook in this folder (`EVAL_POOL_KEY_SEED=999`) -- the natural way to check whether this notebook's own denoising signature (bullet above) still holds on a population that isn't tied to this sweep's own `d0=200,000` pool.

## 10. Does `d0` change how recovery degrades with `noise_viab`?

Sections 1-9 above fix `d0=200,000` and only sweep `noise_viab`. This section adds `d0` as a
SECOND swept variable: for each of the SAME `NOISE_GRID` values, test `5` different `d0` values
-- `D0_GRID_2D = [20,000, 50,000, 100,000, 200,000, 500,000]` (includes this notebook's own
`200,000` baseline as one point, for a direct sanity-check against sections 7-9's numbers).

**Design**: OUTER loop over `d0` (each gets a FRESH pool + `ProtocolV3`, `N1 = mu * d0 / rho` so
`mu=50` stays fixed regardless of `d0` -- same compensation trick as `diversity_sweep.ipynb`),
INNER loop over `noise_viab` (mutating the SAME reused `protocol` object in place, same
advancing-PRNG-key convention as section 6's 1D sweep). `40` total `(d0, noise_viab)`
combinations, each training a fresh `ProfileMLP` -- meaningfully more expensive than the 1D
sweep (`8` points); `batch_size` is adaptive (`max(256, n_train // 50)`, same as
`diversity_sweep.ipynb`) since `d0` varies.

**Top-1000 recovery (10.1) is measured ONLY on the fixed 50,000-sequence eval pool**
(`rec_gt_mlp_eval`, reusing the already-computed `eval_sequences`/`X_eval`/`eval_viab_score` from
section 2 -- free, just one more inference call per point, no extra simulation), NOT on the
in-sweep test fold -- that fold's size scales with `d0` here (`20,000` to `500,000`), which would
make recovery numbers not comparable across the `d0` axis this section is specifically about
(same reasoning as `diversity_sweep.ipynb`/`diversity_sweep_adaptive_D.ipynb`, which dropped
their own in-sweep-test-fold recovery plots for the same reason). Pearson `r` (10.2) still uses
the in-sweep test fold, since `r` doesn't have this degeneracy/comparability problem.

In [ ]:
D0_GRID_2D = [20_000, 50_000, 100_000, 200_000, 500_000]

records_2d = []
base_key_2d = jax.random.key(1)

for d0_val in tqdm(D0_GRID_2D, desc="d0 (outer)", position=0):
    pool_key_2d  = jax.random.fold_in(base_key_2d, d0_val)
    sequences_2d = jax.random.randint(pool_key_2d, shape=(d0_val, NUM_POSITIONS),
                                       minval=0, maxval=NUM_AMINO_ACIDS)
    N1_2d = MU_FIXED * d0_val / RHO_REF
    print(f"for d0 = {d0_val}, N1 = {N1_2d}")

    protocol_2d = ProtocolV3(multinomialNGS=True, N0=N1_2d*10, N1=N1_2d,
            dilution_factor=10, sequences=sequences_2d, D=1e9,
            F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
            noise_viab=0.5, noise_sel=0.5, T_sel=1, T_viab=T_VIAB_FIXED,
            )
    protocol_2d._rho = float(RHO_REF)
    viab_score_2d = np.array(protocol_2d.compute_score(F_viab, J_viab))

    X_all_2d = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[np.asarray(sequences_2d)].reshape(d0_val, -1)
    idx_train_2d, idx_test_2d = train_test_split(np.arange(d0_val), test_size=0.5, random_state=0)
    X_train_full_2d, X_test_2d = X_all_2d[idx_train_2d], X_all_2d[idx_test_2d]
    viab_score_test_2d = viab_score_2d[idx_test_2d]
    batch_size_2d = max(256, len(idx_train_2d) // 50)

    for nv in tqdm(NOISE_GRID, desc=f"noise_viab (d0={d0_val:,})", leave=False, position=1):
        protocol_2d.noise_viab = float(nv)
        protocol_2d.lambda0 = jnp.full(protocol_2d.d0, protocol_2d.N0 / protocol_2d.d0)

        _bio_row_2d, ngs_row_2d = protocol_2d.loop_DE()
        lambda0p_2d, lambda2p_2d, _lambda3p_2d = (np.asarray(a) for a in ngs_row_2d)
        target1_2d = np.log((lambda2p_2d + eps) / (lambda0p_2d + eps))
        target1_train_2d, target1_test_2d = target1_2d[idx_train_2d], target1_2d[idx_test_2d]

        Xtr_2d, ytr_2d, Xva_2d, yva_2d = split_train_val(X_train_full_2d, target1_train_2d,
                                                          val_frac=0.15, seed=0)
        model_2d, val_mse_2d = train_profile_mlp(Xtr_2d, ytr_2d, Xva_2d, yva_2d, epochs=300,
                                                  patience=20, batch_size=batch_size_2d, verbose=False)
        pred_test_2d = np.asarray(predict_log_enrichment(model_2d, jnp.asarray(X_test_2d)))
        pred_eval_2d = np.asarray(predict_log_enrichment(model_2d, jnp.asarray(X_eval)))

        records_2d.append(dict(
            d0=d0_val,
            noise_viab=nv,
            r_gt_protocol=pearson(viab_score_test_2d, target1_test_2d),
            r_gt_mlp=pearson(viab_score_test_2d, pred_test_2d),
            r_protocol_mlp=pearson(target1_test_2d, pred_test_2d),
            rec_gt_mlp_eval=topk_recovery(eval_viab_score, pred_eval_2d, k=K_TOPK),
        ))

noise_d0_grid_df = pd.DataFrame(records_2d)
noise_d0_grid_df

### 91 minutes of computation on the DGX spark

### 10.1 Top-1000 recovery vs `noise_viab`, one line per `d0` (fixed eval pool)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
cmap = plt.get_cmap("viridis", len(D0_GRID_2D))
for i, d0_val in enumerate(D0_GRID_2D):
    sub = noise_d0_grid_df[noise_d0_grid_df["d0"] == d0_val].sort_values("noise_viab")
    ax.plot(sub["noise_viab"], 100 * sub["rec_gt_mlp_eval"], "o-", color=cmap(i), label=f"d0={d0_val:,}")
ax.set_xlabel("noise_viab")
ax.set_ylabel(f"Top-{K_TOPK} recovery (%) -- fixed 50,000-sequence eval pool")
ax.set_title("GT <-> MLP recovery vs noise_viab, by d0 (common benchmark)")
ax.legend(fontsize=8)
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### 10.2 Pearson correlation vs `noise_viab`, one line per `d0`

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for i, d0_val in enumerate(D0_GRID_2D):
    sub = noise_d0_grid_df[noise_d0_grid_df["d0"] == d0_val].sort_values("noise_viab")
    ax.plot(sub["noise_viab"], sub["r_gt_mlp"], "o-", color=cmap(i), label=f"d0={d0_val:,}")
ax.set_xlabel("noise_viab")
ax.set_ylabel("Pearson r (GT <-> MLP, in-sweep test fold)")
ax.set_title("Pearson r vs noise_viab, by d0")
ax.legend(fontsize=8)
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### How to read this (section 10)

- **Does a bigger `d0` shift the curve UP (better recovery at the SAME `noise_viab`), or just
  make it flatter?** Both are "denoising" in a loose sense, but they mean different things: a
  vertical shift means more training data helps at every noise level uniformly; a flatter slope
  means more data specifically helps the model resist HIGH noise better (steeper curve = more
  noise-sensitive), which is the more interesting "denoising via redundancy" signature.
- **Sanity check**: the `d0=200,000` line here should be in the same ballpark as section 9's own
  single-`d0` `rec_gt_mlp_eval` curve (same `noise_viab` grid, same fixed eval pool) -- if it
  doesn't, something diverged between the 1D and 2D sweep setups (different random draws for the
  training pool are expected to cause SOME difference, just not a large one).